In [ ]:
#imports
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# load data
df = pd.read_excel("../data/raw/Base_coronavirus_31-05-2021.xlsx")
df.head()

In [ ]:
df.shape

# Duplicados

## Filas duplicadas

In [ ]:
duplicados = df[df.duplicated(keep=False)] #obtenemos los duplicados

duplicados = duplicados.sort_values(by=df.columns.tolist()) #ordena los duplicados

duplicados.head(10) #muestra los primeros duplicados

In [ ]:
print(df.duplicated().sum())

# Unificar tipos de datos

In [ ]:
print(df.columns)

In [ ]:
# tipos de datos Numericas (Continuas, discretas)
# ["id","edad","peso","talla","ubigeo","infeccion","temperatura","ubigeo_trabajo"]
df_continuas = df[["edad","peso","talla","temperatura"]]
df_discretas = df[["id","ubigeo","ubigeo_trabajo","ano","semana"]]

print(f"Variables continuas:")
df_continuas.info(verbose=True, show_counts=True)
print(f"Variables discretas:")
df_discretas.info(verbose=True, show_counts=True)

In [ ]:
# categoricas (nominales, ordinales)

df_nominales = df[["diresa","red","microred","establecimiento","institucion","entrada","sexo","nacionalidad","pais_nacionalidad","migrante","pais_origen","pais_residencia","departamento_residencia","provincia_residencia","residencia","etnia","departamento_infeccion","provincia_infeccion","distrito_infeccion","hospitalizado","eess","seguro","diagnostico_ing","aislamiento","convulsion_hos","disnea_hos","coma_hos","auscultacion_hos","radiografia_hos","ecografia_hos","tomografia_hos","rmn_hos","otro_hos","servicio_otro","ventilacion","entubado","neumonia","hora_def","lugar_def","fiebre","malestar","tos","garganta","congestion","respiratoria","diarrea","nauseas","cefalea","irritabilidad","muscular","abdominal","pecho","articulaciones","anosmia","ageusia","oido","otros_sintomas","exudado","conjuntival","convulsion","coma","disnea","auscultacion","rxpulmonar","ecografia","tomografia","rmn","otros_signos","embarazo","trimestre","postparto","cardiovascular","diabetes","hepatica","neurologica","inmunodeficiencia","renal","hepatico","pulmonar","cancer","obesidad","tbc","asma","otros_comorbilidad","ocupacion","otra_ocupacion","profesion","diresa_trabajo","red_trabajo","microred_trabajo","eess_trabajo","departamento_trabajo","provincia_trabajo","distrito_trabajo","viajado_14","pais_1","ciudad_1","pais_2","ciudad_2","pais_3","ciudad_3","eess_14","esalud_14","contacto_14","contacto_salud","contacto_familiar","contacto_trabajo","contacto_desconocido","contacto_otro","confirmado_14","entorno_salud","entorno_familiar","entorno_trabajo","casa_reposo","centro_penitenciario","albergue","entorno_desconocido","entorno_otro","contacto_pais","departamento_contacto","provincia_contacto","contacto_ubigeo","contacto_localidad","mercado","mercado_pais","departamento_mercado","provincia_mercado","contacto_mercado","mercado_localidad","muestra","prueba","resultado","muestra1","prueba1","resultado1","muestra2","prueba2","resultado2","muestra_rap","prueba_rap","resultado_rap","muestra_rap1","prueba_rap1","resultado_rap1","fecha_res_rap1","secuenciamiento","asintomatico"]]
df_ordinales = df[["clasificacion","servicio","evolucion"]]

print(f"Variables categoricas nominales:")
df_nominales.info(verbose=True, show_counts=True)
print(f"Variables categoricas ordinales:")
df_ordinales.info(verbose=True, show_counts=True)

In [ ]:
# temporales (Series de tiempo, Duraciones)
df_temp_series = df[["fecha_not","fecha_det","fecha_ini","fecha_res_rap1","fecha_rap1","fecha_res_rap","fecha_rap","fecha_res2","fecha_mue2","fecha_res1","fecha_mue1","fecha_res","fecha_mue","culmina_embarazo","fecha_def","fecha_alt","fecha_ais","fecha_hos","fecha_ini","fecha_det","fecha_not"]]

print(f"Variables temporales (series de tiempo):")
df_temp_series.info(verbose=True, show_counts=True)

In [ ]:
for columna in df.columns:
    # 1. Cálculos de métricas
    valores_unicos_array = df[columna].unique()  # Mantiene los nulos (NaN) por defecto
    contador_valores_unicos = len(valores_unicos_array)
    porcentaje_valores_unicos = (contador_valores_unicos / len(df)) * 100

    # 2. Tomar solo los primeros n valores únicos
    muestra_unicos = valores_unicos_array[:5]

    # 3. Impresión estructurada
    print(f"\n{'='*50}")
    print(f"COLUMNA: {columna}")
    print(f"Total únicos: {contador_valores_unicos} ({porcentaje_valores_unicos:.2f}% del dataset)")
    print("-" * 50)
    print(f"Muestra (5 de {contador_valores_unicos} valores únicos):")
    print(muestra_unicos)
    print("*" * 50)

In [ ]:
# helpers
def print_unique_columns_by_type (df, title=''):
    alto = max(5, len(df.columns) * 0.35)
    resumen = pd.DataFrame({
    'columna': df.columns,
    'tipo': df.dtypes.astype(str),
    'unicos': [df[col].nunique(dropna=False) for col in df.columns]
    }).sort_values(by='unicos', ascending=False)

    plt.figure(figsize=(10, alto))
    sns.barplot(data=resumen, x='unicos', y='columna', hue='tipo', dodge=False)
    plt.title('Cantidad de Valores Únicos por Columna (Agrupados por Tipo)' + title)
    plt.xlabel('Número de Valores Únicos')
    plt.ylabel('Columna')
    plt.tight_layout()
    plt.show()

In [ ]:


# Dataframe de resumen de tipos y valores únicos
print_unique_columns_by_type(df_continuas, ' continuas')
print_unique_columns_by_type(df_discretas, ' discretas')
print_unique_columns_by_type(df_nominales, ' nominales')
print_unique_columns_by_type(df_ordinales, ' ordinales')
# print_unique_columns_by_type(df_temp_series)

In [ ]:
# Calcular proporciones por columna
total_filas = len(df)
datos_composicion = []
alto = max(5, len(df.columns) * 0.35)

for col in df.columns:
    nulos = df[col].isna().sum()
    unicos = df[col].nunique(dropna=True)
    repetidos = total_filas - unicos - nulos

    datos_composicion.append({
        'columna': col,
        'tipo': str(df[col].dtype),
        '% Únicos': (unicos / total_filas) * 100,
        '% Repetidos': (repetidos / total_filas) * 100,
        '% Nulos': (nulos / total_filas) * 100
    })

df_comp = pd.DataFrame(datos_composicion).set_index('columna')
df_comp[['% Únicos', '% Repetidos', '% Nulos']].plot(
    kind='barh', stacked=True, figsize=(10, alto), color=['#2ca02c', '#1f77b4', '#d62728']
)
plt.title('Composición de Valores por Columna')
plt.xlabel('Porcentaje (%)')
plt.tight_layout()
plt.show()

In [ ]:
df = df.drop(columns=['id'])

In [ ]:
# revisar variable objetivo
print(df['evolucion'].value_counts(dropna=False))
print(df['lugar_def'].value_counts(dropna=False))

df['_def_fecha'] = df['fecha_def'].notna() & (df['fecha_def'] != '00-00-0000')
print(df['_def_fecha'].value_counts())

# ¿coinciden las tres fuentes?
print(pd.crosstab(df['_def_fecha'], df['lugar_def'].notna()))
print(pd.crosstab(df['_def_fecha'], df['evolucion'], dropna=False))
print(pd.crosstab(df['_def_fecha'], df['clasificacion'], dropna=False))

In [ ]:
# Variable objetivo: fallecido si cualquiera de las 3 fuentes lo indica
fecha_def_valida = df['fecha_def'].notna() & (df['fecha_def'] != '00-00-0000')
df['fallecido'] = (fecha_def_valida
                   | df['lugar_def'].notna()
                   | (df['evolucion'] == 'FALLECIÓ')).astype(int)
df = df.drop(columns=['_def_fecha'])

# Revisar casos contradictorios
conflictos = df[fecha_def_valida & (df['evolucion'].notna()) & (df['evolucion'] != 'FALLECIÓ')]
print(f"Conflictos: {len(conflictos)}")
display(conflictos[['clasificacion','fecha_ini','fecha_not','fecha_hos','fecha_def',
                    'fecha_alt','lugar_def','evolucion','hospitalizado']])

# Filtrar población: solo casos confirmados
df_conf = df[df['clasificacion'] == 'CONFIRMADO'].copy()
print(df_conf.shape)
print(df_conf['fallecido'].value_counts())
print(df_conf['fallecido'].value_counts(normalize=True).round(4))

In [ ]:
# Eliminar etiquetas ambiguas: fallecido por fecha pero RECUPERADO y sin lugar de defunción
ambiguos = ((df_conf['fallecido'] == 1)
            & (df_conf['evolucion'] == 'RECUPERADO')
            & (df_conf['lugar_def'].isna()))
print(f"Registros ambiguos eliminados: {ambiguos.sum()}")
df_conf = df_conf[~ambiguos].copy()

# Eliminar columnas de fuga de información (ya se usaron para construir el target)
# y clasificacion, que ahora es constante
cols_fuga = ['fecha_def', 'hora_def', 'lugar_def', 'evolucion', 'fecha_alt',
             'diagnostico_ing', 'clasificacion']
df_conf = df_conf.drop(columns=cols_fuga)

print(df_conf.shape)
print(df_conf['fallecido'].value_counts())
print(df_conf['fallecido'].value_counts(normalize=True).round(4))

In [ ]:
grupos = {
    'target':         ['fallecido'],
    'demograficas':   ['edad', 'tipo_edad', 'sexo', 'etnia', 'ocupacion', 'red', 'institucion'],
    'fechas':         ['fecha_ini', 'fecha_not', 'fecha_hos'],   # solo para derivar variables
    'sintomas':       ['fiebre', 'malestar', 'tos', 'garganta', 'congestion', 'respiratoria',
                       'diarrea', 'nauseas', 'cefalea', 'irritabilidad', 'muscular',
                       'abdominal', 'pecho', 'articulaciones', 'anosmia', 'ageusia', 'asintomatico'],
    'signos':         ['temperatura', 'disnea', 'auscultacion', 'rxpulmonar', 'exudado',
                       'conjuntival', 'convulsion', 'coma'],
    'comorbilidades': ['embarazo', 'postparto', 'cardiovascular', 'diabetes', 'hepatica', 'hepatico',
                       'neurologica', 'inmunodeficiencia', 'renal', 'pulmonar', 'cancer',
                       'obesidad', 'asma', 'tbc'],
    'hospitalizacion':['hospitalizado', 'ventilacion', 'entubado', 'neumonia', 'servicio'],
}

cols_sel = [c for g in grupos.values() for c in g]
faltantes = [c for c in cols_sel if c not in df_conf.columns]
assert not faltantes, f"Columnas no encontradas: {faltantes}"

df_model = df_conf[cols_sel].copy()
print(df_model.shape)

# Resumen de nulos y tipos por grupo
resumen = pd.DataFrame({
    'grupo': [g for g, cols in grupos.items() for _ in cols],
    'tipo': df_model.dtypes.astype(str).values,
    '% nulos': (df_model.isna().mean() * 100).round(2).values,
    'unicos': df_model.nunique().values,
}, index=cols_sel)
display(resumen)

In [ ]:
# 1. ¿Los bloques de nulos dependen de la fecha (cambio de ficha)?
mes = pd.to_datetime(df_model['fecha_not'], format='%d-%m-%Y', errors='coerce').dt.to_period('M')
display(pd.DataFrame({
    'n': df_model.groupby(mes).size(),
    '% nulo bloque 28%': df_model['temperatura'].isna().groupby(mes).mean().mul(100).round(1),
    '% nulo bloque 65%': df_model['obesidad'].isna().groupby(mes).mean().mul(100).round(1),
    '% fallecidos': df_model.groupby(mes)['fallecido'].mean().mul(100).round(2),
}))

# 2. Variables de hospitalización vs hospitalizado
print(df_model['hospitalizado'].value_counts(dropna=False))
for c in ['ventilacion', 'entubado', 'neumonia', 'servicio']:
    print(f"\n--- {c} ---")
    print(pd.crosstab(df_model['hospitalizado'], df_model[c].fillna('NaN')))

# 3. asintomatico vs síntomas registrados
sintomas = ['fiebre', 'malestar', 'tos', 'garganta', 'congestion', 'respiratoria', 'diarrea',
            'nauseas', 'cefalea', 'irritabilidad', 'muscular', 'abdominal', 'pecho', 'articulaciones']
n_sint = df_model[sintomas].sum(axis=1)
print(pd.crosstab(df_model['asintomatico'].fillna('NaN'), n_sint.clip(upper=3).rename('n_sintomas (3=3+)')))

# 4. hepatica vs hepatico
print(pd.crosstab(df_model['hepatica'], df_model['hepatico'].fillna(-1).rename('hepatico (-1=NaN)')))

# 5. Otras categóricas a revisar
for c in ['tipo_edad', 'etnia', 'ocupacion', 'institucion']:
    print(f"\n{df_model[c].value_counts(dropna=False)}")

In [ ]:
print(pd.crosstab(df_model['hospitalizado'], df_model['neumonia'].fillna('NaN')).to_string())
print(pd.crosstab(df_model['hospitalizado'], df_model['servicio'].fillna('NaN')).to_string())
print(pd.crosstab(df_model['asintomatico'].fillna('NaN'), n_sint.clip(upper=3)).to_string())
print(pd.crosstab(df_model['hepatica'], df_model['hepatico'].fillna(-1)).to_string())
for c in ['tipo_edad', 'etnia', 'ocupacion']:
    print(df_model[c].value_counts(dropna=False).to_string(), '\n')

# Limpieza

In [ ]:
df_clean = df_model.copy()
sintomas = ['fiebre', 'malestar', 'tos', 'garganta', 'congestion', 'respiratoria', 'diarrea',
            'nauseas', 'cefalea', 'irritabilidad', 'muscular', 'abdominal', 'pecho', 'articulaciones']

# ---------- 1. Fechas y variables temporales ----------
for c in ['fecha_not', 'fecha_ini', 'fecha_hos']:
    df_clean[c] = pd.to_datetime(df_clean[c].replace('00-00-0000', np.nan),
                                 format='%d-%m-%Y', errors='coerce')

antes = len(df_clean)
df_clean = df_clean[df_clean['fecha_not'] >= '2020-03-01'].copy()
print(f"Eliminados por fecha_not anterior a mar-2020: {antes - len(df_clean)}")

# Ola por fecha de notificación (coincide con el cambio de ficha)
df_clean['ola'] = np.where(df_clean['fecha_not'] < '2020-12-01', 1, 2)

# Retraso inicio de síntomas -> notificación; valores implausibles a NaN
# (la imputación se hará dentro del pipeline, después del train/test split)
dias = (df_clean['fecha_not'] - df_clean['fecha_ini']).dt.days
df_clean['dias_ini_not'] = dias.where(dias.between(0, 60))
print(f"dias_ini_not nulos o implausibles: {df_clean['dias_ini_not'].isna().sum()}")

# ---------- 2. Edad en años ----------
factor = df_clean['tipo_edad'].map({'AÑOS': 1, 'MES': 1/12, 'DIA': 1/365})
df_clean['edad'] = (df_clean['edad'] * factor).round(2)

# ---------- 3. Síntomas y comorbilidades ----------
for c in ['obesidad', 'asma', 'tbc', 'anosmia', 'ageusia']:   # solo existen en la ficha nueva
    df_clean[c] = df_clean[c].map({1: 'SI', 0: 'NO'}).fillna('NO_REGISTRADO')

df_clean['hepatica'] = ((df_clean['hepatica'] == 1) | (df_clean['hepatico'] == 1)).astype(int)
df_clean['n_sintomas'] = df_clean[sintomas].sum(axis=1)

# ---------- 4. Hospitalización ----------
corr_hosp = df_clean['servicio'].notna() & (df_clean['hospitalizado'] != 'SI')
print(f"hospitalizado corregido a SI por tener servicio: {corr_hosp.sum()}")
df_clean.loc[corr_hosp, 'hospitalizado'] = 'SI'
no_hosp = df_clean['hospitalizado'] == 'NO'

df_clean.loc[no_hosp & (df_clean['ventilacion'] == 'SI'), 'ventilacion'] = 'DESCONOCIDO'
for c in ['ventilacion', 'entubado', 'neumonia']:
    df_clean.loc[no_hosp & df_clean[c].isna(), c] = 'NO'
    df_clean[c] = df_clean[c].fillna('DESCONOCIDO')

df_clean['servicio'] = df_clean['servicio'].map({
    'UNIDAD DE CUIDADOS INTENSIVOS - UCI': 'UCI',
    'SALA DE AISLAMIENTO': 'AISLAMIENTO',
    'OTRA AREA DE SERVICIO': 'OTRA_AREA'})
df_clean.loc[no_hosp, 'servicio'] = 'NO_HOSPITALIZADO'
df_clean['servicio'] = df_clean['servicio'].fillna('DESCONOCIDO')

# ---------- 5. Categóricas ----------
df_clean['etnia'] = (df_clean['etnia']
                     .replace({'Afrodescendiente': 'Otro', 'Asiático descendiente': 'Otro'})
                     .fillna('NO_REGISTRADO'))
df_clean['ocupacion'] = df_clean['ocupacion'].replace({
    'TRABAJADOR DE SALUD EN LABORATORIO': 'TRABAJADOR DE SALUD',
    'POLICIA': 'POLICIA/MILITAR', 'MILITAR': 'POLICIA/MILITAR',
    'TRABAJA CON ANIMALES': 'OTROS'})
df_clean['institucion'] = df_clean['institucion'].fillna('DESCONOCIDO')

# ---------- 6. Eliminar columnas ya procesadas o descartadas ----------
df_clean = df_clean.drop(columns=['temperatura', 'coma', 'hepatico', 'asintomatico',
                                  'tipo_edad', 'fecha_ini', 'fecha_hos'])
# fecha_not se conserva para el EDA temporal; se elimina antes de modelar

# ---------- 7. Verificación ----------
print(df_clean.shape)
nulos = df_clean.isna().sum()
print("Columnas con nulos:\n", nulos[nulos > 0])
print(pd.crosstab(df_clean['ola'], df_clean['fallecido'], margins=True))
print(df_clean['edad'].describe())

In [ ]:
# Reconstruir fecha_ini original para diagnosticar
fi = pd.to_datetime(df_model.loc[df_clean.index, 'fecha_ini'].replace('00-00-0000', np.nan),
                    format='%d-%m-%Y', errors='coerce')
dias = (df_clean['fecha_not'] - fi).dt.days

estado = pd.Series(np.select([fi.isna(), dias < 0, dias > 60],
                             ['fecha_ini nula', 'negativo', 'mayor a 60'], 'valido'),
                   index=df_clean.index, name='estado')

print(pd.crosstab(estado, df_clean['ola'], margins=True).to_string(), '\n')
print((pd.crosstab(estado, df_clean['fallecido'], normalize='index') * 100).round(2).to_string(), '\n')
print("Negativos:\n", dias[dias < 0].describe().to_string(), '\n')
print("Mayores a 60:\n", dias[dias > 60].describe().to_string(), '\n')
print("Válidos:\n", dias[dias.between(0, 60)].describe().to_string())

In [ ]:
print(pd.crosstab(estado, df_clean['n_sintomas'].clip(upper=3).rename('n_sintomas (3=3+)'),
                  normalize='index').mul(100).round(1).to_string(), '\n')
print(pd.crosstab(estado, df_clean['hospitalizado'], normalize='index').mul(100).round(1).to_string())

# Checkpoint (parquet conserva los tipos, incluidas las fechas)
df_clean.to_csv('../data/interim/covid_puno_clean.csv', index=False)

In [ ]:
semanal = (df_clean.set_index('fecha_not').resample('W')['fallecido']
           .agg(casos='count', fallecidos='sum'))
semanal['letalidad'] = (semanal['fallecidos'] / semanal['casos'] * 100).where(semanal['casos'] >= 30)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].bar(semanal.index, semanal['casos'], width=5, alpha=0.6, label='Casos confirmados')
axes[0].set_ylabel('Casos')
ax2 = axes[0].twinx()
ax2.plot(semanal.index, semanal['fallecidos'], color='red', marker='o', ms=3, label='Fallecidos')
ax2.set_ylabel('Fallecidos')
axes[0].set_title('Curva epidémica semanal - COVID-19 Puno (casos confirmados)')
fig.legend(loc='upper left', bbox_to_anchor=(0.07, 0.93))

axes[1].plot(semanal.index, semanal['letalidad'], color='black', marker='o', ms=3)
axes[1].set_ylabel('Letalidad semanal (%)')
axes[1].set_title('Letalidad semanal (semanas con 30 o más casos)')

for ax in axes:
    ax.axvline(pd.Timestamp('2020-12-01'), ls='--', color='gray')
plt.tight_layout()
plt.show()

In [ ]:
num = ['edad', 'dias_ini_not', 'n_sintomas']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, c in enumerate(num):
    sns.histplot(data=df_clean, x=c, hue='fallecido', stat='density',
                 common_norm=False, bins=30, ax=axes[0, i])
    sns.boxplot(data=df_clean, x='fallecido', y=c, ax=axes[1, i])
    axes[0, i].set_title(f'Distribución de {c}')
plt.tight_layout()
plt.show()

display(df_clean.groupby('fallecido')[num].describe().T.round(2))

# Letalidad por grupo de edad
grupo_edad = pd.cut(df_clean['edad'], bins=[0, 18, 30, 40, 50, 60, 70, 80, 101], right=False)
tabla_edad = df_clean.groupby(grupo_edad, observed=True)['fallecido'].agg(casos='count', fallecidos='sum')
tabla_edad['letalidad_%'] = (tabla_edad['fallecidos'] / tabla_edad['casos'] * 100).round(2)
display(tabla_edad)

In [ ]:
fall = df.loc[df_clean.index][df_clean['fallecido'] == 1]
fdef = pd.to_datetime(fall['fecha_def'].replace('00-00-0000', np.nan), format='%d-%m-%Y', errors='coerce')
fnot = pd.to_datetime(fall['fecha_not'], format='%d-%m-%Y', errors='coerce')
t_muerte = (fdef - fnot).dt.days
print(t_muerte.describe(percentiles=[.5, .75, .9, .95]).round(1).to_string())
print("Casos notificados por semana en mayo 2021:")
print(df_clean[df_clean['fecha_not'] >= '2021-05-01'].set_index('fecha_not')
      .resample('W')['fallecido'].agg(['count', 'sum']).to_string())

In [ ]:
corte_censura = pd.Timestamp('2021-05-15')
n_antes, f_antes = len(df_clean), df_clean['fallecido'].sum()

df_clean = df_clean[df_clean['fecha_not'] <= corte_censura].copy()

print(f"Registros eliminados por censura: {n_antes - len(df_clean)} "
      f"({(n_antes - len(df_clean)) / n_antes * 100:.2f}%)")
print(f"Fallecidos eliminados: {f_antes - df_clean['fallecido'].sum()}")
print(df_clean.shape)
print(df_clean['fallecido'].value_counts(normalize=True).mul(100).round(2).to_string())
print(pd.crosstab(df_clean['ola'], df_clean['fallecido'], margins=True).to_string())

# Actualizar el checkpoint
df_clean.to_csv('../data/interim/covid_puno_clean.csv', index=False)

In [ ]:
excluir = ['fallecido', 'ola']
binarias = [c for c in df_clean.columns if c not in excluir
            and set(df_clean[c].dropna().unique()) <= {0, 1}]

filas = []
for c in binarias:
    g = df_clean.groupby(c)['fallecido'].agg(['count', 'mean'])
    filas.append({'variable': c, 'n_con_condicion': g.loc[1, 'count'],
                  'letal_con_%': g.loc[1, 'mean'] * 100, 'letal_sin_%': g.loc[0, 'mean'] * 100})
rr = pd.DataFrame(filas)
rr['RR'] = rr['letal_con_%'] / rr['letal_sin_%']
rr = rr.sort_values('RR').round(2)
display(rr)

plt.figure(figsize=(9, 8))
plt.barh(rr['variable'], rr['RR'], color=np.where(rr['RR'] > 1, '#d62728', '#1f77b4'))
plt.axvline(1, color='black', ls='--')
plt.xscale('log')
plt.xlabel('Riesgo relativo de fallecer (escala log)')
plt.title('Riesgo relativo por síntoma, signo y comorbilidad')
plt.tight_layout()
plt.show()

# Categóricas: letalidad por categoría
categoricas = ['sexo', 'etnia', 'ocupacion', 'red', 'institucion', 'obesidad', 'asma', 'tbc',
               'anosmia', 'ageusia', 'hospitalizado', 'ventilacion', 'entubado', 'neumonia', 'servicio']
for c in categoricas:
    t = df_clean.groupby(c)['fallecido'].agg(casos='count', fallecidos='sum')
    t['letalidad_%'] = (t['fallecidos'] / t['casos'] * 100).round(2)
    print(f"\n--- {c} ---\n{t.sort_values('letalidad_%', ascending=False).to_string()}")

In [ ]:
df_fe = df_clean.copy()

# 1. Recuento de comorbilidades
comorb = ['cardiovascular', 'diabetes', 'hepatica', 'neurologica',
          'inmunodeficiencia', 'renal', 'pulmonar', 'cancer']
df_fe['n_comorbilidades'] = df_fe[comorb].sum(axis=1) + (df_fe['obesidad'] == 'SI').astype(int)

# 2. Fusionar anosmia y ageusia (ambas son NO_REGISTRADO en la ficha antigua)
df_fe['anosmia_ageusia'] = np.select(
    [df_fe['anosmia'] == 'NO_REGISTRADO', (df_fe['anosmia'] == 'SI') | (df_fe['ageusia'] == 'SI')],
    ['NO_REGISTRADO', 'SI'], default='NO')

# 3. Eliminar variables muy raras y categorías mínimas
df_fe = df_fe.drop(columns=['anosmia', 'ageusia', 'asma', 'tbc', 'postparto'])
df_fe = df_fe[(df_fe['red'] != 'SIN RED') & (df_fe['institucion'] != 'DESCONOCIDO')].copy()
df_fe['institucion'] = df_fe['institucion'].replace({
    'SANIDAD DE LA POLICIA NACIONAL DEL PERU': 'SANIDAD PNP/FFAA',
    'SANIDAD DEL EJERCITO DEL PERU': 'SANIDAD PNP/FFAA'})
print(df_fe.shape)

t = df_fe.groupby(df_fe['n_comorbilidades'].clip(upper=3))['fallecido'].agg(casos='count', fallecidos='sum')
t['letalidad_%'] = (t['fallecidos'] / t['casos'] * 100).round(2)
print(t.rename(index={3: '3+'}).to_string(), '\n')

# 4. Riesgo relativo crudo vs ajustado por edad (Mantel-Haenszel)
grupo_edad = pd.cut(df_fe['edad'], bins=[0, 18, 30, 40, 50, 60, 70, 80, 101], right=False)

def rr_crudo_y_mh(expo, y=df_fe['fallecido']):
    crudo = y[expo == 1].mean() / y[expo == 0].mean()
    num = den = 0
    for _, idx in df_fe.groupby(grupo_edad, observed=True).groups.items():
        e, yy, N = expo.loc[idx], y.loc[idx], len(idx)
        num += yy[e == 1].sum() * (e == 0).sum() / N
        den += yy[e == 0].sum() * (e == 1).sum() / N
    return crudo, (num / den if den > 0 else np.nan)

exposiciones = {c: df_fe[c] for c in ['respiratoria', 'disnea', 'auscultacion', 'rxpulmonar',
                                      'fiebre', 'tos', 'malestar', 'embarazo'] + comorb}
exposiciones['sexo_masculino'] = (df_fe['sexo'] == 'MASCULINO').astype(int)

filas = [{'variable': k, 'n_expuestos': int(v.sum()), **dict(zip(['RR_crudo', 'RR_ajustado_edad'],
          rr_crudo_y_mh(v)))} for k, v in exposiciones.items()]
tabla_mh = pd.DataFrame(filas)
tabla_mh['cambio_%'] = ((tabla_mh['RR_ajustado_edad'] / tabla_mh['RR_crudo'] - 1) * 100)
display(tabla_mh.sort_values('RR_ajustado_edad', ascending=False).round(2))

In [ ]:
from scipy.stats import chi2_contingency

def pares_altos(m, umbral):
    tri = m.where(np.triu(np.ones(m.shape, dtype=bool), k=1)).stack()
    return tri[tri.abs() >= umbral].sort_values(key=abs, ascending=False).round(2)

# --- Numéricas y binarias: Spearman ---
num_bin = [c for c in df_fe.select_dtypes('number').columns if c != 'fallecido']
corr = df_fe[num_bin].corr(method='spearman')

plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True, cbar_kws={'shrink': .7})
plt.title('Correlación de Spearman - variables numéricas y binarias')
plt.tight_layout()
plt.show()
print("Pares con |r| >= 0.4:\n", pares_altos(corr, 0.4).to_string(), '\n')

# --- Categóricas: V de Cramér ---
def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2 = chi2_contingency(tabla, correction=False)[0]
    return np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))

df_cat = df_fe.select_dtypes(exclude=['number', 'datetime']).assign(ola=df_fe['ola'].astype(str))
cats = df_cat.columns.tolist()
cv = pd.DataFrame(np.eye(len(cats)), index=cats, columns=cats)
for i, a in enumerate(cats):
    for b in cats[i + 1:]:
        cv.loc[a, b] = cv.loc[b, a] = cramers_v(df_cat[a], df_cat[b])

plt.figure(figsize=(10, 8))
sns.heatmap(cv.astype(float), cmap='Reds', vmin=0, vmax=1, annot=True, fmt='.2f', square=True)
plt.title('V de Cramér - variables categóricas')
plt.tight_layout()
plt.show()
print("Pares con V >= 0.5:\n", pares_altos(cv.astype(float), 0.5).to_string())

In [ ]:
df_fe.to_csv('../data/interim/covid_puno_features.csv', index=False)

In [ ]:
import json
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

RUTA = Path('../data/processed')
RUTA.mkdir(parents=True, exist_ok=True)
df_fe = pd.read_csv('../data/interim/covid_puno_features.csv')

# ---------- Columnas ----------
sintomas = ['fiebre', 'malestar', 'tos', 'garganta', 'congestion', 'respiratoria', 'diarrea',
            'nauseas', 'cefalea', 'irritabilidad', 'muscular', 'abdominal', 'pecho', 'articulaciones']
signos = ['disnea', 'auscultacion', 'rxpulmonar', 'exudado', 'conjuntival', 'convulsion']
comorb = ['cardiovascular', 'diabetes', 'hepatica', 'neurologica', 'inmunodeficiencia',
          'renal', 'pulmonar', 'cancer', 'embarazo']
num_cols = ['edad', 'dias_ini_not', 'n_sintomas', 'n_comorbilidades']
bin_cols = sintomas + signos + comorb
cat_base = ['sexo', 'etnia', 'ocupacion', 'red', 'institucion', 'obesidad', 'anosmia_ageusia', 'ola']
cat_hosp = ['ventilacion', 'entubado', 'neumonia', 'servicio']

def construir_conjunto(df, incluir_hosp):
    cats = cat_base + (cat_hosp if incluir_hosp else [])
    X = df[num_cols + bin_cols + cats].copy()
    X[cats] = X[cats].astype(str)
    return X, df['fallecido'], cats

def dividir(X, y):
    estrato = y.astype(str) + '_' + X['ola']
    return train_test_split(X, y, test_size=0.2, stratify=estrato, random_state=42)

conjuntos = {}
for nombre, df_sub, hosp in [('triaje', df_fe, False),
                             ('hospitalario', df_fe[df_fe['hospitalizado'] == 'SI'], True)]:
    X, y, cats = construir_conjunto(df_sub, hosp)
    Xtr, Xte, ytr, yte = dividir(X, y)
    Xtr.assign(fallecido=ytr).to_csv(RUTA / f'{nombre}_train.csv')
    Xte.assign(fallecido=yte).to_csv(RUTA / f'{nombre}_test.csv')
    conjuntos[nombre] = dict(Xtr=Xtr, ytr=ytr, cats=cats)
    print(f"{nombre}: train {Xtr.shape}, test {Xte.shape} | "
          f"% fallecidos train {ytr.mean()*100:.2f}, test {yte.mean()*100:.2f}")

In [ ]:
def pasos_preprocesamiento(cats, usar_smote=True):
    imputar = ColumnTransformer(
        [('dias', SimpleImputer(strategy='median'), ['dias_ini_not'])],
        remainder='passthrough', verbose_feature_names_out=False).set_output(transform='pandas')
    codificar = ColumnTransformer(
        [('num', StandardScaler(), num_cols),
         ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), cats)],
        remainder='passthrough', verbose_feature_names_out=False)
    pasos = [('imputar', imputar)]
    if usar_smote:
        pasos.append(('smote', SMOTENC(categorical_features=bin_cols + cats, random_state=42)))
    pasos.append(('codificar', codificar))
    return pasos

# Preprocesadores SIN ajustar: se usan dentro de la validación cruzada en el modelado
for nombre, smote in [('triaje', True), ('hospitalario', False)]:
    joblib.dump(ImbPipeline(pasos_preprocesamiento(conjuntos[nombre]['cats'], usar_smote=smote)),
                RUTA / f'preprocesador_{nombre}.joblib')

# Train de triaje balanceado (solo para entrenar el modelo final, NO para validación cruzada)
c = conjuntos['triaje']
X_res, y_res = ImbPipeline(pasos_preprocesamiento(c['cats'])[:-1]).fit_resample(c['Xtr'], c['ytr'])
X_res.assign(fallecido=y_res.values).to_csv(RUTA / 'triaje_train_smotenc.csv')
print("Antes de SMOTENC:", c['ytr'].value_counts().to_dict())
print("Después de SMOTENC:", y_res.value_counts().to_dict())

In [ ]:
num_ns = ['edad', 'n_sintomas', 'n_comorbilidades']
bin_ns = ['fiebre', 'tos', 'malestar', 'respiratoria', 'disnea', 'auscultacion', 'rxpulmonar',
          'cardiovascular', 'diabetes', 'renal', 'pulmonar', 'embarazo']
descriptores = ['fallecido', 'ola', 'red', 'hospitalizado', 'etnia', 'ocupacion']

# Versión sin escalar (para K-prototypes u otros métodos de datos mixtos)
X_ns_raw = df_fe[num_ns + bin_ns + ['sexo']].copy()
X_ns_raw.to_csv(RUTA / 'no_supervisado_X_raw.csv')

# Versión escalada (para K-means, jerárquico, DBSCAN, PCA)
X_ns = X_ns_raw.copy()
X_ns['sexo_masculino'] = (X_ns.pop('sexo') == 'MASCULINO').astype(int)
X_ns[num_ns] = MinMaxScaler().fit_transform(X_ns[num_ns])
X_ns.to_csv(RUTA / 'no_supervisado_X.csv')

# Descriptores para caracterizar los clusters (mismo índice)
df_fe[descriptores].to_csv(RUTA / 'no_supervisado_descriptores.csv')
print("No supervisado:", X_ns.shape, "| nulos:", X_ns.isna().sum().sum())

In [ ]:
# ---------- Supervisado: un solo archivo con columna de partición ----------
estrato = (df_fe['fallecido'].astype(str) + '_' + df_fe['ola'].astype(str)
           + '_' + df_fe['hospitalizado'])
idx_train, idx_test = train_test_split(df_fe.index, test_size=0.2,
                                       stratify=estrato, random_state=42)

cols_sup = num_cols + bin_cols + cat_base + cat_hosp + ['hospitalizado', 'fallecido']
df_sup = df_fe[cols_sup].copy()
df_sup['particion'] = 'test'
df_sup.loc[idx_train, 'particion'] = 'train'
df_sup.to_csv(RUTA / 'dataset_supervisado.csv', index_label='id_registro')

print("Supervisado:", df_sup.shape)
print((pd.crosstab(df_sup['particion'], df_sup['fallecido'], normalize='index') * 100).round(2))
hosp = df_sup[df_sup['hospitalizado'] == 'SI']
print((pd.crosstab(hosp['particion'], hosp['fallecido'], normalize='index') * 100).round(2))

# ---------- No supervisado: variables + descriptores en un solo archivo ----------
df_ns = pd.concat([X_ns, df_fe[descriptores].add_prefix('desc_')], axis=1)
df_ns.to_csv(RUTA / 'dataset_no_supervisado.csv', index_label='id_registro')
print("\nNo supervisado:", df_ns.shape, "| nulos en variables:", X_ns.isna().sum().sum())

In [ ]:
from sklearn.ensemble import IsolationForest

# ---------- 1. Outliers estadísticos (IQR y z-score) ----------
filas = []
for c in ['edad', 'dias_ini_not', 'n_sintomas', 'n_comorbilidades']:
    s = df_fe[c].dropna()
    q1, q3 = s.quantile([.25, .75])
    lim_inf, lim_sup = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    es_iqr = (df_fe[c] < lim_inf) | (df_fe[c] > lim_sup)
    es_z = ((df_fe[c] - s.mean()) / s.std()).abs() > 3
    filas.append({'variable': c, 'lim_inf': lim_inf, 'lim_sup': lim_sup,
                  'n_iqr': es_iqr.sum(), '%_iqr': es_iqr.mean() * 100, 'n_z>3': es_z.sum(),
                  'letal_outliers_%': df_fe.loc[es_iqr, 'fallecido'].mean() * 100,
                  'letal_resto_%': df_fe.loc[~es_iqr, 'fallecido'].mean() * 100})
display(pd.DataFrame(filas).round(2))

# ---------- 2. Outliers lógicos (reglas de dominio) ----------
reglas = {
    'embarazo en hombre':            (df_fe['embarazo'] == 1) & (df_fe['sexo'] == 'MASCULINO'),
    'embarazo fuera de 12-50 años':  (df_fe['embarazo'] == 1) & ~df_fe['edad'].between(12, 50),
    'trabajador salud < 18 años':    (df_fe['ocupacion'] == 'TRABAJADOR DE SALUD') & (df_fe['edad'] < 18),
    'policía/militar < 18 años':     (df_fe['ocupacion'] == 'POLICIA/MILITAR') & (df_fe['edad'] < 18),
    'estudiante > 60 años':          (df_fe['ocupacion'] == 'ESTUDIANTE') & (df_fe['edad'] > 60),
    'sanidad PNP/FFAA < 18 años':    (df_fe['institucion'] == 'SANIDAD PNP/FFAA') & (df_fe['edad'] < 18),
    'edad 0 con comorbilidades':     (df_fe['edad'] < 1) & (df_fe['n_comorbilidades'] > 0),
    '12 o más síntomas':             df_fe['n_sintomas'] >= 12,
}
tabla_reglas = pd.DataFrame({
    'n_registros': {k: v.sum() for k, v in reglas.items()},
    'fallecidos': {k: df_fe.loc[v, 'fallecido'].sum() for k, v in reglas.items()}})
display(tabla_reglas)

# ---------- 3. Outliers multivariados (Isolation Forest sobre el perfil clínico) ----------
X_iso = df_ns.drop(columns=df_ns.filter(like='desc_').columns)
iso = IsolationForest(contamination=0.01, random_state=42).fit(X_iso)
anomalo = pd.Series(iso.predict(X_iso) == -1, index=X_iso.index)

print(f"\nRegistros anómalos (1%): {anomalo.sum()}")
print(f"Letalidad anómalos: {df_fe.loc[anomalo, 'fallecido'].mean()*100:.2f}% "
      f"vs resto: {df_fe.loc[~anomalo, 'fallecido'].mean()*100:.2f}%")
perfil = pd.DataFrame({'anomalos': df_fe.loc[anomalo, num_ns + bin_ns].mean(),
                       'resto': df_fe.loc[~anomalo, num_ns + bin_ns].mean()}).round(3)
display(perfil)